In [ ]:
# 1. 라이브러리 임포트

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score, precision_recall_fscore_support
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, Input

In [30]:
# 2. 데이터 불러오기

df = pd.read_csv("scaled.csv")
print(f"원본 데이터 shape: {df.shape}")

원본 데이터 shape: (37836, 31)


In [31]:

# 3. 'Class' 결측치 확인 및 제거

# 3-1. (확인) 'Class' 열에 NaN이 있는지 확인
nan_count = df['Class'].isnull().sum()
if nan_count > 0:
    print(f"경고: 'Class' 열에 {nan_count}개의 NaN(결측치)이 있습니다. 이 행들을 제거합니다.")
    # 3-2. (해결) 'Class' 열에 NaN이 있는 행(row)을 제거
    df.dropna(subset=['Class'], inplace=True)

경고: 'Class' 열에 1개의 NaN(결측치)이 있습니다. 이 행들을 제거합니다.


In [32]:
# 4. 입력(X), 출력(y) 분리

X = df.drop(columns=['Class'])
y = df['Class']

# 이미 scaled.csv는 스케일링 완료된 상태이므로 StandardScaler 불필요

In [33]:
# 5. Train/Test Split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Train/Test 분리 완료")

Train/Test 분리 완료


In [34]:
# 6. 클래스 가중치 계산

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weights = {0: class_weights[0], 1: class_weights[1]}
print(f"클래스 가중치: {class_weights}")

클래스 가중치: {0: np.float64(0.5013582455442921), 1: np.float64(184.5609756097561)}


In [35]:
# 7. MLP 모델 정의

def build_mlp(input_dim):
    model = Sequential([
        Input(shape=(input_dim,)),
        Dense(128, activation='relu'),
        BatchNormalization(),
        Dropout(0.3),

        Dense(64, activation='relu'),
        BatchNormalization(),
        Dropout(0.3),

        Dense(32, activation='relu'),
        Dense(1, activation='sigmoid')
    ])
    model.compile(
        optimizer='adam',
        loss='binary_crossentropy'
    )
    return model

In [36]:
# 8. 모델 학습

model = build_mlp(X_train.shape[1])
history = model.fit(
    X_train, y_train,
    epochs=8,
    batch_size=512,
    validation_split=0.2,
    verbose=1,
    class_weight=class_weights
)

Epoch 1/8
48/48 ━━━━━━━━━━━━━━━━━━━━ 4s 27ms/step - loss: 1.5622 - val_loss: 0.6055
Epoch 2/8
48/48 ━━━━━━━━━━━━━━━━━━━━ 2s 25ms/step - loss: 0.2595 - val_loss: 0.3982
Epoch 3/8
48/48 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.1697 - val_loss: 0.2453
Epoch 4/8
48/48 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.1332 - val_loss: 0.1635
Epoch 5/8
48/48 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.1169 - val_loss: 0.1165
Epoch 6/8
48/48 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.1018 - val_loss: 0.0866
Epoch 7/8
48/48 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.1060 - val_loss: 0.0744
Epoch 8/8
48/48 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0683 - val_loss: 0.0606


In [38]:
# 9. 예측 및 평가

from sklearn.metrics import precision_score, recall_score  # ← 이 줄 추가!

y_pred_prob = model.predict(X_test)
threshold = 0.5
y_pred = (y_pred_prob > threshold).astype(int)

recall = recall_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)

print("\n[모델 평가 결과]")
print(f"Threshold = {threshold}")
print(f"Recall: {recall:.4f}")
print(f"Precision: {precision:.4f}")

237/237 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step

[모델 평가 결과]
Threshold = 0.5
Recall: 0.9524
Precision: 0.1923
